# ROGII Wellbore Geology - LSTM Inference

Load the trained BiLSTM, predict TVT for test wells, generate submission.

**Features (6):** `MD, X, Y, Z, GR, TVT_input`

**Runtime**: Kaggle GPU - **Author**: Samir Attrah

In [ ]:
# Cell 1: Environment & Imports
import os
os.environ["KERAS_BACKEND"] = "jax"

import keras
import jax.numpy as jnp
import numpy as np
import polars as pl
import glob, pickle, warnings
warnings.filterwarnings("ignore")
print(f"Keras: {keras.__version__}, Backend: {keras.backend.backend()}")


In [ ]:
# Cell 2: Auto-detect dataset path

def find_data_dir():
    """Searches common Kaggle mount points for the ROGII dataset.

    Returns:
        Absolute path to the dataset root directory.

    Raises:
        FileNotFoundError: If no valid dataset is found.
    """
    candidates = [
        "/kaggle/input/competitions/rogii-wellbore-geology-prediction",
        "/kaggle/input/rogii-wellbore-geology-prediction",
        "/home/samer/Documents/competitions/ROGII/dataset",
    ]

    for scan_root in ["/kaggle/input", "/kaggle/input/competitions"]:
        if os.path.isdir(scan_root):
            for entry in os.listdir(scan_root):
                full = os.path.join(scan_root, entry)
                if os.path.isdir(full) and full not in candidates:
                    candidates.append(full)

    print("Searching for ROGII test dataset...")
    for path in candidates:
        if not os.path.isdir(path):
            continue

        contents = os.listdir(path)
        has_test = "test" in contents and os.path.isdir(os.path.join(path, "test"))
        
        if has_test:
            n_test = len(glob.glob(os.path.join(path, "test", "*__horizontal_well.csv")))
            if n_test > 0:
                print(f"  V Using {path} (found {n_test} test wells)")
                return path

    raise FileNotFoundError("Could not find ROGII dataset with 'test' directory.")

DATA_DIR = find_data_dir()


In [ ]:
# Cell 3: Auto-detect model artifact

def find_model_dir():
    """Searches Kaggle input paths for the trained model artifact.

    Returns:
        Path to directory containing lstm_tvt_model.keras.

    Raises:
        FileNotFoundError: If no model artifact found.
    """
    print("Searching for model artifact in input directory...")

    # Prioritize /kaggle/input and /kaggle/input/competitions
    candidates = []
    for scan_root in ["/kaggle/input", "/kaggle/input/competitions"]:
        if os.path.isdir(scan_root):
            for entry in os.listdir(scan_root):
                candidates.append(os.path.join(scan_root, entry))
    
    # Local fallbacks (keep them but prioritize Kaggle inputs)
    candidates += [
        "/home/samer/Documents/competitions/ROGII/models",
        "/tmp/rogii_output",
    ]

    for path in candidates:
        if not os.path.isdir(path):
            continue
        keras_files = glob.glob(os.path.join(path, "*.keras"))
        if keras_files:
            print(f"  -> {path}")
            print(f"    .keras files: {[os.path.basename(f) for f in keras_files]}")
            print(f"  V Using this as MODEL_DIR")
            return path
        # Check one level deeper (often datasets are nested)
        try:
            for sub in os.listdir(path):
                sub_path = os.path.join(path, sub)
                if os.path.isdir(sub_path):
                    keras_sub = glob.glob(os.path.join(sub_path, "*.keras"))
                    if keras_sub:
                        print(f"  -> {sub_path}")
                        return sub_path
        except:
            continue

    raise FileNotFoundError(
        "No model artifact (.keras) found in Kaggle input. Checked candidates."
    )

MODEL_DIR = find_model_dir()


In [ ]:
# Cell 4: Configuration

OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle") else "/tmp/rogii_output"
os.makedirs(OUT_DIR, exist_ok=True)

CONFIG = {
    "data_dir": DATA_DIR,
    "model_path": os.path.join(MODEL_DIR, "lstm_tvt_model.keras"),
    "scaler_path": os.path.join(MODEL_DIR, "scaler_params.pkl"),
    "submission_path": os.path.join(OUT_DIR, "submission.csv"),
    "window_size": 64,
}

FEATURE_COLS = [
    "MD", "X", "Y", "Z", "GR", "TVT_input",
]

print(f"Data dir      : {CONFIG['data_dir']}")
print(f"Model path    : {CONFIG['model_path']}")
print(f"Scaler path   : {CONFIG['scaler_path']}")
print(f"Submission out: {CONFIG['submission_path']}")


In [ ]:
# Cell 5: Load model & scaler

print("Loading model...")
model = keras.saving.load_model(CONFIG["model_path"])
print(f"✓ Model loaded: {model.name}, params={model.count_params():,}")
model.summary()

print("\nLoading scaler...")
with open(CONFIG["scaler_path"], "rb") as f:
    scaler = pickle.load(f)
print(f"✓ Scaler loaded: {list(scaler.keys())}")
print(f"  Target mean={scaler['target_mean']:.2f}, std={scaler['target_std']:.2f}")


In [ ]:
# Cell 6: Preprocessing (identical to training)

def preprocess(df):
    """Same preprocessing pipeline as training notebook."""
    for col in ["GR", "TVT_input"]:
        if col in df.columns:
            df = df.with_columns(
                pl.col(col).interpolate()
                  .fill_null(strategy="forward").fill_null(strategy="backward")
                  .fill_null(0.0)
            )
    return df

print("V Preprocessing pipeline defined.")


In [ ]:
# Cell 7: Load submission template & predict

sample_sub = pl.read_csv(os.path.join(CONFIG["data_dir"], "sample_submission.csv"))
sample_sub = sample_sub.with_columns([
    pl.col("id").str.extract(r"^(.+)_(\d+)$", 1).alias("well_id"),
    pl.col("id").str.extract(r"^(.+)_(\d+)$", 2).cast(pl.Int64).alias("row_idx"),
])
print(f"Sample submission: {sample_sub.shape[0]} rows")
print(f"Wells in submission: {sample_sub['well_id'].unique().to_list()}")

test_ids = sorted(
    os.path.basename(f).split("__")[0]
    for f in glob.glob(os.path.join(CONFIG["data_dir"], "test", "*__horizontal_well.csv"))
)
print(f"Test well files found: {test_ids}")

def predict_well(model, df, scaler, ws):
    """Predict TVT for every row using sliding windows."""
    feats = np.nan_to_num(df.select(FEATURE_COLS).to_numpy().astype(np.float32))
    feats_n = (feats - scaler["feat_mean"]) / scaler["feat_std"]
    n = len(feats_n)
    preds = np.full(n, np.nan, dtype=np.float32)
    starts = list(range(0, n - ws + 1))
    if not starts:
        return preds
    X = np.stack([feats_n[s:s+ws] for s in starts])
    yn = model.predict(X, batch_size=512, verbose=0).ravel()
    yp = yn * scaler["target_std"] + scaler["target_mean"]
    for i, s in enumerate(starts):
        preds[s + ws - 1] = yp[i]
    fv = ws - 1
    if fv < n and not np.isnan(preds[fv]):
        preds[:fv] = preds[fv]
    return preds

all_preds = {}
for wid in test_ids:
    print(f"\nPredicting well: {wid}")
    df = preprocess(pl.read_csv(
        os.path.join(CONFIG["data_dir"], "test", f"{wid}__horizontal_well.csv"),
        infer_schema_length=10000))
    print(f"  Loaded {len(df)} rows, columns: {df.columns}")
    all_preds[wid] = predict_well(model, df, scaler, CONFIG["window_size"])
    valid = np.sum(~np.isnan(all_preds[wid]))
    print(f"  Predicted {valid}/{len(all_preds[wid])} rows, "
          f"TVT range: [{np.nanmin(all_preds[wid]):.1f}, {np.nanmax(all_preds[wid]):.1f}]")


In [ ]:
# Cell 8: Build, validate & save submission

rows = []
for r in sample_sub.iter_rows(named=True):
    wid, idx = r["well_id"], r["row_idx"]
    arr = all_preds.get(wid)
    val = float(arr[idx]) if arr is not None and idx < len(arr) and not np.isnan(arr[idx]) else 0.0
    rows.append({"id": r["id"], "tvt": val})

submission = pl.DataFrame(rows)
assert submission.shape[0] == sample_sub.shape[0], "Row count mismatch!"
assert set(submission["id"].to_list()) == set(sample_sub["id"].to_list()), "ID mismatch!"

submission.select(["id", "tvt"]).write_csv(CONFIG["submission_path"])
print(f"\n✅ Submission saved: {CONFIG['submission_path']}")
print(f"   Rows: {submission.shape[0]}")
print(f"   TVT mean={submission['tvt'].mean():.2f}, std={submission['tvt'].std():.2f}")
print(submission.head(10))


In [ ]:
# Cell 9: Visualise predictions

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(test_ids), figsize=(6*len(test_ids), 5), squeeze=False)
for i, wid in enumerate(test_ids):
    axes[0,i].plot(all_preds[wid], lw=0.6)
    axes[0,i].set_title(f"Well {wid}")
    axes[0,i].set_xlabel("Row"); axes[0,i].set_ylabel("TVT")
plt.suptitle("Predicted TVT — Test Wells"); plt.tight_layout(); plt.show()
